# II - Extraction tryptiques depuis un fichier `.conllu`

Ce notebook :
- lit un fichier `.conllu`
- extrait pour chaque verbe :
  - la **phrase**
  - le **sujet**
  - le **verbe**
  - l’**objet / complément**
  - les **dépendances** associées
- exporte le résultat en **CSV**


## 1) Paramètres

Adaptez si besoin le chemin du `.conllu` et le chemin de sortie du `.csv`.


In [1]:
import pandas as pd
import os

INPUT_CONLLU = os.environ.get("CONLLU_INPUT")
OUTPUT_CSV = os.environ.get("TRIPTYQUES_OUTPUT")

## 2) Fonctions

Règles utilisées :
- **Sujet direct** : `nsubj`, `nsubj:pass`
- **Héritage du sujet** si le verbe est en `xcomp`, `ccomp`, `advcl`
- **Relative** `acl:relcl` : on prend le nom tête comme sujet implicite
- **Coordination** `conj` : on hérite du sujet du verbe gouverneur
- **Objets / compléments** : `obl:arg`, `obl:mod`, `obl:agent`, `obj`, `iobj`, `expl:comp`
- **Verbe prédicat** : on remonte si la tête est en `xcomp`, `ccomp`, `conj`, `advcl`


In [2]:
def parse_custom_conllu(path):
    """ 
    Entrée : (str) un chemin de fichier .conllu
    Sortie : (list) une liste de dictionnaires avec toutes les informations du conllu
    """

    sentences = []
    comments = []
    tokens = []

    with open(path, "r", encoding="utf-8") as file:
        for raw_line in file:

            line = raw_line.rstrip("\n")

            # si on tombe sur une ligne vide, on ajoute les infos a sentences et on repart de 0
            if not line.strip():
                if tokens:
                    sentences.append({"comments": comments, "tokens": tokens}) 
                comments, tokens = [], []
                continue

            if line.startswith("#"):
                comments.append(line) # dans un CONLLU les commentaires commencent par un #, c'est la ou l'on note les infos de phrases plus globales
                continue

            parts = line.split("\t") # une ligne valide de conllu doit avoir 10 "parties" séparés d'une tabulation (\t)
            
            tok = {
                "id": int(parts[0]), # ID du token
                "form": parts[1], # forme du mot
                "lemma": parts[2], # lemme
                "upos": parts[3], # POS tag
                "xpos": parts[4], 
                "feats_raw": parts[5], # informations comme le genre par exemple
                "head": int(parts[6]) if parts[6] != "_" else 0, # head syntaxique, si jamais il n'y en a pas on assigne la valeur 0
                "deprel": parts[7], # relation de dépendance (spacy)
                "col8": parts[8],   # annotation Propp / coréference
                "col9": parts[9],   # éntités nommées Propp : PER, FAC, LOC, etc.
                "doc_id": int(parts[10]) if len(parts) > 10 and parts[10] != "_" else pd.NA,
                "align_id": int(parts[11]) if len(parts) > 11 and parts[11] != "_" else pd.NA,
            }
            
            tokens.append(tok)

    if tokens:
        sentences.append({"comments": comments, "tokens": tokens})

    return sentences


def join_sentence_forms(tokens):
    """
    Entrée : (dict) un dictionnaire avec les informations d'une seule phrase (venant du fichier conllu)
    Sortie : (str) retourne la phrase en texte brut, en tenant compte des espaces apres la ponctuation
    """
    forms = []
 
    PUNCT_NO_SPACE_AFTER = {"(", "[", "{", "«"}

    for tok in tokens:
        form = tok.get("form", "") 

        if forms and tok.get("upos") == "PUNCT":
            forms[-1] = forms[-1] + form
        elif forms and forms[-1] in PUNCT_NO_SPACE_AFTER:
            forms[-1] = forms[-1] + form
        else:
            forms.append(form)

    return " ".join(forms).strip()


def get_comment_value(comments, prefix):
    """
    Entrée : (list) commentaires d'une phrase du conllu
             (str) préfixe d'un commentaire
    Sortie : (str) commentaire, ici numéro de phrase ou numéro de paragraphe
    """
    for c in comments:
        if c.startswith(prefix):
            return c.split("=", 1)[1].strip()   # recupère ce qu'il y a après le premier =
    return "NaN"


def token_by_id(tokens):
    return {tok["id"]: tok for tok in tokens}


def children_map(tokens):
    children = {}
    for tok in tokens:
        children.setdefault(tok["head"], []).append(tok)
    return children


def group_token_ids(head_id, tokens):
    """
    Renvoie les IDs des tokens du groupe autour d'une tête syntaxique.
    Exemple : tête = maison ; groupe = la maison de Saville-row.
    
    Cette fonction retourne les tokens du groupe puis est utilisée pour avoir son texte avec build_group().
    """
    if pd.isna(head_id):
        return []

    head_id = int(head_id)

    by_id = token_by_id(tokens)
    children = children_map(tokens)
    collected = set()

    def rec(tok_id):
        if tok_id in collected or tok_id not in by_id:
            return

        tok = by_id[tok_id]

        if tok["upos"] == "VERB":
            return
        if tok["deprel"] == "punct":
            return

        collected.add(tok_id)

        for child in children.get(tok_id, []):
            rec(child["id"])

    rec(head_id)

    return sorted(collected)


def build_group(head_id, tokens):
    """
    Construit le texte du groupe à partir des IDs récupérés par group_token_ids().
    """
    if pd.isna(head_id):
        return "NaN"

    ids = group_token_ids(head_id, tokens)

    if not ids:
        return "NaN"

    by_id = token_by_id(tokens)
    all_ids = {tok["id"] for tok in tokens}

    # On complète les groupes du type Reform - Club ou Saville - row.
    # Si le groupe contient Reform, on ajoute éventuellement "-" et "Club".
    # Si le groupe contient Club, on ajoute éventuellement "Reform" et "-".
    # Si le groupe contient "-", on ajoute éventuellement les deux côtés.
    expanded_ids = set(ids)

    for i in list(ids):
        tok = by_id.get(i)
        if tok is None:
            continue

        # cas ou le groupe commence par Reform
        if i + 1 in all_ids and i + 2 in all_ids:
            next_tok = by_id[i + 1]
            next_next_tok = by_id[i + 2]

            if next_tok["form"] in ["-", "–", "—"]:
                if next_next_tok["upos"] in ["NOUN", "PROPN", "ADJ"]:
                    expanded_ids.add(i + 1)
                    expanded_ids.add(i + 2)

        # cas : Reform - Club, quand le groupe commence sur Club
        if i - 1 in all_ids and i - 2 in all_ids:
            prev_tok = by_id[i - 1]
            prev_prev_tok = by_id[i - 2]

            if prev_tok["form"] in ["-", "–", "—"]:
                if prev_prev_tok["upos"] in ["NOUN", "PROPN", "ADJ"]:
                    expanded_ids.add(i - 1)
                    expanded_ids.add(i - 2)

        # cas où la tête elle-même est le tiret
        if tok["form"] in ["-", "–", "—"]:
            if i - 1 in all_ids:
                expanded_ids.add(i - 1)
            if i + 1 in all_ids:
                expanded_ids.add(i + 1)

    ids = sorted(expanded_ids)
    forms = [by_id[i]["form"] for i in ids]

    PUNCT_NO_SPACE_BEFORE = {".", ",", ";", ":", "!", "?", ")", "]", "}", "%", "»", "…"}
    PUNCT_NO_SPACE_AFTER = {"(", "[", "{", "«"}
    DASHES = {"-", "–", "—"}

    out = []

    for form in forms:

        # ponctuation qui se colle au mot précédent
        if form in PUNCT_NO_SPACE_BEFORE and out:
            out[-1] = out[-1] + form

        # tiret interne : Reform - Club -> Reform-
        elif form in DASHES and out:
            out[-1] = out[-1] + form

        # mot après un tiret ou après une apostrophe : Reform- Club -> Reform-Club ; l' année -> l'année
        elif out and (out[-1].endswith(tuple(DASHES)) or out[-1].endswith("'") or out[-1].endswith("’")):
            out[-1] = out[-1] + form

        # mot après une ponctuation ouvrante : ( mot -> (mot
        elif out and out[-1] in PUNCT_NO_SPACE_AFTER:
            out[-1] = out[-1] + form

        else:
            out.append(form)

    return " ".join(out).strip()


def find_subject(verb, tokens):
    
    # Sujet direct
    for tok in tokens:
        if tok["head"] == verb["id"] and tok["deprel"] in ["nsubj", "nsubj:pass"]:
            return tok

    # Héritage xcomp / ccomp / advcl
    if verb["deprel"] in ["xcomp", "ccomp", "advcl"]:
        head_id = verb["head"]
        for tok in tokens:
            if tok["head"] == head_id and tok["deprel"] in ["nsubj", "nsubj:pass"]:
                return tok

    # Relative
    if verb["deprel"] == "acl:relcl":
        head_id = verb["head"]
        for tok in tokens:
            if tok["id"] == head_id and tok["upos"] in ["NOUN", "PROPN", "PRON"]:
                return tok

    # Coordination
    if verb["deprel"] == "conj":
        head_id = verb["head"]
        head_tok = next((tok for tok in tokens if tok["id"] == head_id), None)
        if head_tok is not None:
            return find_subject(head_tok, tokens)

    return None


def find_predicate_verb(head_id, tokens):
    # Tête directement verbale
    for tok in tokens:
        if tok["id"] == head_id and tok["upos"] == "VERB":
            return tok

    # Remontée si xcomp / ccomp / conj / advcl
    for tok in tokens:
        if tok["id"] == head_id:
            if tok["deprel"] in ["xcomp", "ccomp", "conj", "advcl"] and tok["head"] != 0:
                return find_predicate_verb(tok["head"], tokens)

    return None


def find_objects_for_verb(verb, tokens):
    targets = ["obl:arg", "obl:mod", "obl:agent", "obj", "iobj", "expl:comp"]
    objs = []

    for tok in tokens:
        if tok["deprel"] in targets:
            pred = find_predicate_verb(tok["head"], tokens)
            if pred is not None and pred["id"] == verb["id"]:
                objs.append(tok)

    objs.sort(key=lambda x: x["id"])
    return objs

## 3) Extraction et export CSV

In [3]:
def build_svo_dataframe(input_conllu):
    sentences = parse_custom_conllu(input_conllu)
    rows = []

    for sent_index, sent in enumerate(sentences, start=1):

        # on récupère la partie sans les commentaires
        tokens = sent["tokens"]
        phrase_txt = join_sentence_forms(tokens)

        # puis les commentaires qui nous interessent : le numéro de phrase et de paragraphe
        num_phrase = get_comment_value(sent["comments"], "# Num phrase ")
        num_paragr = get_comment_value(sent["comments"], "# Num paragr ")

        for verb in tokens:
            if verb["upos"] != "VERB":  # on itere sur tous les tokens donc on séléctionne seulement ceux qui sont des verbes
                continue

            subj = find_subject(verb, tokens)

            id_sujet = subj["id"] if subj else pd.NA
            sujet = build_group(id_sujet, tokens)
            dep_sujet = subj["deprel"] if subj else "NaN"

            objs = find_objects_for_verb(verb, tokens)

            if not objs:
                objs = [None]

            by_id = token_by_id(tokens)

            for obj in objs:
                id_objet = obj["id"] if obj else pd.NA
                objet = build_group(id_objet, tokens)
                dep_objet = obj["deprel"] if obj else "NaN"

                align_id_sujet = by_id[id_sujet]["align_id"] if pd.notna(id_sujet) else pd.NA
                align_id_verbe = verb["align_id"]
                align_id_objet = by_id[id_objet]["align_id"] if pd.notna(id_objet) else pd.NA

                doc_id_sujet = by_id[id_sujet]["doc_id"] if pd.notna(id_sujet) else pd.NA
                doc_id_verbe = verb["doc_id"]
                doc_id_objet = by_id[id_objet]["doc_id"] if pd.notna(id_objet) else pd.NA

                rows.append({
                    "Phrase": phrase_txt,
                    "Sujet": sujet,
                    "Verbe": verb["form"],
                    "Objet": objet,

                    "Dep_sujet": dep_sujet,
                    "Dep_verbe": verb["deprel"],
                    "Dep_objet": dep_objet,

                    "ID_sujet": id_sujet,
                    "ID_verbe": verb["id"],
                    "ID_objet": id_objet,

                    "DocID_sujet": doc_id_sujet,
                    "DocID_verbe": doc_id_verbe,
                    "DocID_objet": doc_id_objet,

                    "AlignID_sujet": align_id_sujet,
                    "AlignID_verbe": align_id_verbe,
                    "AlignID_objet": align_id_objet,

                    "IDs_sujet": group_token_ids(id_sujet, tokens),
                    "IDs_objet": group_token_ids(id_objet, tokens),

                    "Lemme_verbe": verb["lemma"],
                    "Head_verbe": verb["head"],

                    "Annotation_verbe_col8": verb["col8"],
                    "Annotation_verbe_col9": verb["col9"],

                    "Num_phrase": num_phrase,
                    "Num_paragr": num_paragr,
                    "Sentence_index": sent_index,
                })

    return pd.DataFrame(rows), sentences


df_svo, sentences = build_svo_dataframe(INPUT_CONLLU)
df_svo.head(10)

,Phrase,Sujet,Verbe,Objet,Dep_sujet,Dep_verbe,Dep_objet,ID_sujet,ID_verbe,ID_objet,...,AlignID_objet,IDs_sujet,IDs_objet,Lemme_verbe,Head_verbe,Annotation_verbe_col8,Annotation_verbe_col9,Num_phrase,Num_paragr,Sentence_index
0,"En l' année 1872, la maison portant le numéro ...",NaN,portant,le numéro 7 de Saville-row Burlington Gardens-...,NaN,acl,obj,<NA>,8,10,...,9,[],"[9, 10, 11, 12, 13, 14, 15, 17, 18, 20]",porter,7,5,FAC,0,0,1
1,"En l' année 1872, la maison portant le numéro ...",Sheridan,mourut,en 1814,nsubj,acl:relcl,obl:mod,23,24,26,...,23,[23],"[25, 26]",mourir,20,_,_,0,0,1
2,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,En l'année 1872,nsubj:pass,root,obl:mod,7,30,3,...,2,"[6, 7]","[1, 2, 3, 4]",habiter,0,20,_,0,0,1
3,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,par Phileas Fogg esq.,nsubj:pass,root,obl:agent,7,30,32,...,28,"[6, 7]","[31, 32, 33, 35, 36]",habiter,0,20,_,0,0,1
4,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,l'un des membres les plus singuliers et les pl...,nsubj:pass,root,obl:mod,7,30,39,...,35,"[6, 7]","[38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 4...",habiter,0,20,_,0,0,1
5,"En l' année 1872, la maison portant le numéro ...",la maison,semblât,NaN,nsubj:pass,advcl,NaN,7,59,<NA>,...,<NA>,"[6, 7]",[],sembler,30,_,_,0,0,1
6,"En l' année 1872, la maison portant le numéro ...",NaN,prendre,à tâche,NaN,xcomp,obl:arg,<NA>,60,62,...,57,[],"[61, 62]",prendre,59,_,_,0,0,1
7,"En l' année 1872, la maison portant le numéro ...",NaN,faire,NaN,NaN,acl,NaN,<NA>,66,<NA>,...,<NA>,[],[],faire,62,_,_,0,0,1
8,"En l' année 1872, la maison portant le numéro ...",qui,pût,NaN,nsubj,acl:relcl,NaN,67,68,<NA>,...,<NA>,[67],[],pouvoir,62,_,_,0,0,1
9,"En l' année 1872, la maison portant le numéro ...",qui,attirer,l'attention,nsubj,xcomp,obj,67,69,71,...,66,[67],"[70, 71]",attirer,68,_,_,0,0,1


## 4) Des tryptiques au mouvement

Objectif de cette partie : maintenant que l'on a les tryptiques SUJET/VERBE/OBJET d'un texte en .csv, réutilisons les entités nommés stockées dans le .conllu correspondant pour avoir des tryptiques PERSONNAGE/VERBE/LIEU (ou dans l'autre sens).

In [4]:
# Fonction utilitaire : garder les valeurs uniques en conservant l'ordre
def unique_list(values):
    out = []
    for v in values:
        if pd.notna(v) and v != "_" and v not in out:
            out.append(v)
    return out


# Nettoyer les catégories Propp / SACR
# Exemple : "p PER", "PER", "p PER_VERBTYPE=" deviennent "PER"
def clean_cat(cat):
    cat = str(cat)

    if "PER" in cat:
        return "PER"
    if "FAC" in cat:
        return "FAC"
    if "LOC" in cat:
        return "LOC"

    return cat


# Chercher les entités présentes dans une liste d'IDs de tokens
# tokens = tous les tokens de la phrase
# ids = IDs du groupe sujet ou du groupe objet
# wanted_cats = catégories recherchées, par exemple {"PER"} ou {"FAC", "LOC"}
def entities_in_ids(tokens, ids, wanted_cats):
    texts = []
    cats = []

    for tok in tokens:
        # On regarde seulement les tokens qui appartiennent au groupe sujet/objet
        if tok["id"] in ids:
            cat = clean_cat(tok["col9"])

            # Si la catégorie du token est celle qu'on cherche, on garde l'annotation
            if cat in wanted_cats:
                texts.append(tok["col8"])
                cats.append(cat)

    return unique_list(texts), unique_list(cats)

## Ajout des colonnes

In [5]:

# Dictionnaire permettant de retrouver les tokens d'une phrase
# Sentence_index dans df_svo commence à 1, comme enumerate(..., start=1)
sent_by_index = {
    i + 1: sent["tokens"]
    for i, sent in enumerate(sentences)
}

# Création des nouvelles colonnes vides dans df_svo
df_svo["personnages_sujet"] = [[] for _ in range(len(df_svo))]
df_svo["personnages_objet"] = [[] for _ in range(len(df_svo))]
df_svo["personnages_triptyque"] = [[] for _ in range(len(df_svo))]

df_svo["lieux_sujet"] = [[] for _ in range(len(df_svo))]
df_svo["lieux_objet"] = [[] for _ in range(len(df_svo))]
df_svo["lieux_triptyque"] = [[] for _ in range(len(df_svo))]
df_svo["types_lieux_triptyque"] = [[] for _ in range(len(df_svo))]


# Parcours de chaque triptyque
for idx, row in df_svo.iterrows():

    # On récupère les tokens de la phrase d'origine du triptyque
    tokens = sent_by_index[row["Sentence_index"]]

    # Personnages dans le groupe sujet et dans le groupe objet
    pers_sujet, _ = entities_in_ids(tokens, row["IDs_sujet"], {"PER"})
    pers_objet, _ = entities_in_ids(tokens, row["IDs_objet"], {"PER"})

    # Lieux dans le groupe sujet et dans le groupe objet
    lieux_sujet, types_lieux_sujet = entities_in_ids(tokens, row["IDs_sujet"], {"FAC", "LOC"})
    lieux_objet, types_lieux_objet = entities_in_ids(tokens, row["IDs_objet"], {"FAC", "LOC"})

    # Agrégation au niveau du triptyque
    personnages = unique_list(pers_sujet + pers_objet)
    lieux = unique_list(lieux_sujet + lieux_objet)
    types_lieux = unique_list(types_lieux_sujet + types_lieux_objet)

    # Remplissage des colonnes personnage
    df_svo.at[idx, "personnages_sujet"] = pers_sujet
    df_svo.at[idx, "personnages_objet"] = pers_objet
    df_svo.at[idx, "personnages_triptyque"] = personnages

    # Remplissage des colonnes lieu
    df_svo.at[idx, "lieux_sujet"] = lieux_sujet
    df_svo.at[idx, "lieux_objet"] = lieux_objet
    df_svo.at[idx, "lieux_triptyque"] = lieux
    df_svo.at[idx, "types_lieux_triptyque"] = types_lieux


# Variables booléennes utiles pour filtrer ensuite
df_svo["a_personnage_triptyque"] = df_svo["personnages_triptyque"].apply(lambda x: len(x) > 0)
df_svo["a_lieu_triptyque"] = df_svo["lieux_triptyque"].apply(lambda x: len(x) > 0)

df_svo["a_personnage_et_lieu_triptyque"] = (
    df_svo["a_personnage_triptyque"] 
    & df_svo["a_lieu_triptyque"]
)

## Bilan

In [6]:
# Résumé rapide
print("Nombre total de triptyques :", len(df_svo))
print("Avec personnage :", df_svo["a_personnage_triptyque"].sum())
print("Avec lieu FAC/LOC :", df_svo["a_lieu_triptyque"].sum())
print("Avec personnage ET lieu :", df_svo["a_personnage_et_lieu_triptyque"].sum())


# Affichage de contrôle
df_svo[
    [
        "Phrase",
        "Sujet",
        "Verbe",
        "Objet",
        "IDs_sujet",
        "IDs_objet",
        "personnages_triptyque",
        "lieux_triptyque",
        "types_lieux_triptyque",
        "a_personnage_et_lieu_triptyque",
    ]
].head(10)

Nombre total de triptyques : 1255
Avec personnage : 675
Avec lieu FAC/LOC : 128
Avec personnage ET lieu : 61


,Phrase,Sujet,Verbe,Objet,IDs_sujet,IDs_objet,personnages_triptyque,lieux_triptyque,types_lieux_triptyque,a_personnage_et_lieu_triptyque
0,"En l' année 1872, la maison portant le numéro ...",NaN,portant,le numéro 7 de Saville-row Burlington Gardens-...,[],"[9, 10, 11, 12, 13, 14, 15, 17, 18, 20]",[],"[5, 10]",[FAC],False
1,"En l' année 1872, la maison portant le numéro ...",Sheridan,mourut,en 1814,[23],"[25, 26]",[],[],[],False
2,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,En l'année 1872,"[6, 7]","[1, 2, 3, 4]",[],[5],[FAC],False
3,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,par Phileas Fogg esq.,"[6, 7]","[31, 32, 33, 35, 36]",[0],[5],[FAC],True
4,"En l' année 1872, la maison portant le numéro ...",la maison,habitée,l'un des membres les plus singuliers et les pl...,"[6, 7]","[38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 4...",[0],"[5, 3]",[FAC],True
5,"En l' année 1872, la maison portant le numéro ...",la maison,semblât,NaN,"[6, 7]",[],[],[5],[FAC],False
6,"En l' année 1872, la maison portant le numéro ...",NaN,prendre,à tâche,[],"[61, 62]",[],[],[],False
7,"En l' année 1872, la maison portant le numéro ...",NaN,faire,NaN,[],[],[],[],[],False
8,"En l' année 1872, la maison portant le numéro ...",qui,pût,NaN,[67],[],[],[],[],False
9,"En l' année 1872, la maison portant le numéro ...",qui,attirer,l'attention,[67],"[70, 71]",[],[],[],False


## Sauvegarde

In [7]:
df_svo.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print("CSV créé :", OUTPUT_CSV)

CSV créé : ../results/csv_triptyques/manuel_chap1to5.csv
